# 🔮 Day 146 — Streamlit ML Explainability Dashboard
## Month 8 | Deployment & App Building | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 146 (Month 8, Week 2, Day 1) |
| **Topic** | ML explainability in Streamlit · SHAP visualization · Batch prediction · Plotly performance charts |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverable** | `day146_app/` — 3-page ML explainability Streamlit app |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

> **Why this matters for freelancing:**
> Any data scientist can train a model. Very few can explain it interactively.
> An ML explainability dashboard — where a client can enter inputs, see a prediction,
> AND understand *why* — commands ₹15k–50k on freelance platforms.
> SHAP values (Month 7) + Streamlit (Month 8) = one of the rarest skill combos in Indian freelancing.

---

## Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| **146** | **ML Explainability Dashboard** | **Today** |


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** `seed=141`, 500 rows, 11 columns.
> This is the same FreelanceHub India dataset used throughout Month 8.


In [ ]:
# ─── RAW DATA GENERATOR ───────────────────────────────────────────────────
# Purpose : Reproduce Month 8 FreelanceHub India dataset (seed=141, 500 rows)
# Method  : Seeded random generation with realistic distribution priors

import pandas as pd
import numpy as np

np.random.seed(141)
n = 500

categories = ['ML/AI', 'Web Dev', 'Data Analytics', 'Graphic Design',
              'Content Writing', 'SEO/Digital Marketing']
platforms  = ['Upwork', 'Freelancer', 'Fiverr', 'Toptal', 'Direct Client']
cities     = ['Mumbai', 'Bangalore', 'Delhi', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
levels     = ['Entry', 'Mid', 'Senior']
outcomes   = ['Completed', 'Cancelled', 'In Progress']

df = pd.DataFrame({
    'project_id':    [f'PJ{1000+i}' for i in range(n)],
    'category':      np.random.choice(categories, n),
    'platform':      np.random.choice(platforms, n, p=[0.30, 0.25, 0.25, 0.10, 0.10]),
    'city':          np.random.choice(cities, n),
    'level':         np.random.choice(levels, n, p=[0.40, 0.40, 0.20]),
    'budget_inr':    np.random.randint(5000, 120001, n),
    'duration_days': np.random.randint(7, 91, n),
    'client_rating': np.round(np.random.uniform(2.5, 5.0, n), 1),
    'bids_received': np.random.randint(2, 51, n),
    'outcome':       np.random.choice(outcomes, n, p=[0.65, 0.20, 0.15]),
    'month_posted':  np.random.choice(['Jan','Feb','Mar','Apr','May','Jun',
                                       'Jul','Aug','Sep','Oct','Nov','Dec'], n),
})

# Introduce realistic nulls
null_idx = np.random.choice(n, 25, replace=False)
df.loc[null_idx[:10],  'client_rating'] = np.nan
df.loc[null_idx[10:20],'bids_received'] = np.nan
df.loc[null_idx[20:],  'duration_days'] = np.nan

print(f"Shape: {df.shape}")
print(df.dtypes)
print()
print(df.head(5).to_string())


---
## 📚 Section 2 — Concept Notes

### What is ML Explainability?

A prediction alone has **zero client value** if the client can't trust it.  
Explainability bridges the gap: it shows *which features drove* the prediction and *by how much*.

---

### SHAP (SHapley Additive exPlanations) — Quick Recap

| SHAP Concept | What it tells you |
|---|---|
| **Global importance** | Mean `|SHAP|` per feature across all test rows — which features matter most *on average* |
| **Local explanation** | SHAP values for one specific prediction — why *this particular* row got this result |
| **Positive SHAP** | Feature pushed prediction *toward* the positive class |
| **Negative SHAP** | Feature pushed prediction *away* from positive class |

```python
# Global SHAP (already covered in Month 7)
explainer  = shap.TreeExplainer(model)
shap_vals  = explainer.shap_values(X_test)   # list[2] for binary RF
shap_pos   = shap_vals[1]                    # class = 1 (Completed)
mean_shap  = np.abs(shap_pos).mean(axis=0)   # shape: (n_features,)
```

---

### New Streamlit Concepts for Day 146

#### 1. `@st.cache_resource` — Load model ONCE
```python
@st.cache_resource
def load_model():
    return joblib.load("day146_model.pkl")
```
> **Rule:** `@st.cache_data` = cacheable data (DataFrames, CSVs)
> `@st.cache_resource` = non-serializable objects (ML models, DB connections)

#### 2. `st.columns([ratio_list])` — Asymmetric layouts
```python
col1, col2, col3 = st.columns([2, 1, 1])   # 2:1:1 width ratio
with col1:
    st.metric("Accuracy", "62%")
```

#### 3. `st.plotly_chart()` — Interactive Plotly in Streamlit
```python
import plotly.express as px
fig = px.bar(df, x="feature", y="importance", title="Feature Importance")
st.plotly_chart(fig, use_container_width=True)
```
> Plotly charts support hover, zoom, pan — matplotlib charts do not.
> For client demos, always prefer Plotly.

#### 4. `st.download_button()` — Let users download results
```python
csv_bytes = results_df.to_csv(index=False).encode('utf-8')
st.download_button(
    label="⬇ Download Predictions",
    data=csv_bytes,
    file_name="predictions.csv",
    mime="text/csv"
)
```

#### 5. Confidence colour coding
```python
# Threshold-based visual feedback
if prob >= 0.70:
    st.success(f"✅ Completed — {prob*100:.1f}% confidence")
elif prob >= 0.50:
    st.warning(f"⚠️ Likely Completed — {prob*100:.1f}% confidence")
else:
    st.error(f"❌ Not Completed — {(1-prob)*100:.1f}% confidence")
```

---

### App Architecture for Day 146

```
day146_app/
├── Home.py                          ← Model overview + scorecard
├── pages/
│   ├── 1_🔮_Single_Prediction.py   ← Single project prediction + explanation
│   ├── 2_📊_Batch_Prediction.py    ← CSV upload → predict all → download
│   └── 3_📈_Model_Performance.py   ← Confusion matrix + ROC curve + SHAP
└── day146_model.pkl                 ← Saved RandomForest bundle
```

---

### Confusion Matrix Interpretation

| | Predicted: Not Completed | Predicted: Completed |
|---|---|---|
| **Actual: Not Completed** | TN = 0 | FP = 35 |
| **Actual: Completed** | FN = 3 | TP = 62 |

> **Day 146 model has TN=0:** it almost never predicts "Not Completed."
> This is a *class imbalance problem* — 65% of data is Completed.
> The model learned to always predict the majority class.
> **NRA action:** This should be flagged in the app, not hidden.


---
## 🛠 Section 3 — Practice Guide

**Deliverable:** `day146_app/` folder with 4 Python files + saved model  
**Run command:** `streamlit run day146_app/Home.py`

---

### Task 1 (10 pts) — Feature Engineering + Model Training + Save

**Goal:** Prepare FreelanceHub data, train a RandomForest, save as `day146_model.pkl`

**File:** `build_model.py` (run once to generate the pkl, NOT part of the Streamlit app)

**Steps:**
1. Load the FreelanceHub data using `np.random.seed(141)`
2. Create `target = 1 if outcome=='Completed' else 0`
3. Fill nulls: `client_rating` with median, `bids_received` with median, `duration_days` with median
4. Label-encode: `category`, `platform`, `level` (use `LabelEncoder`, save each encoder)
5. Features: `budget_inr`, `duration_days`, `client_rating`, `bids_received`, `category_enc`, `platform_enc`, `level_enc`
6. Train/test split: `test_size=0.20`, `random_state=141`, `stratify=y`
7. Train `RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=141)`
8. Save bundle: `joblib.dump({'model':model, 'le_dict':le_dict, 'feat_cols':feat_cols, 'medians':medians}, 'day146_model.pkl')`

**Answer key:** Accuracy = **62.00%**, ROC-AUC = **0.5108**

---

### Task 2 (10 pts) — Home.py — Model Overview Page

**Goal:** Build the app entry point with model summary and key metrics

**Steps:**
1. `st.set_page_config(page_title="FreelanceHub Predictor", page_icon="🔮", layout="wide")`
2. Load model bundle using `@st.cache_resource`
3. Display title, subtitle, and a 4-column metric row:
   - Accuracy: `62%`
   - ROC-AUC: `0.51`
   - Training rows: `400`
   - Test rows: `100`
4. Add an `st.info()` box explaining what the app does in one sentence
5. Add an `st.warning()` box noting: *"Model AUC = 0.51 — near-random. Use predictions with caution. This dashboard is a demonstration of ML deployment patterns."*

---

### Task 3 (10 pts) — Single Prediction Form

**File:** `pages/1_🔮_Single_Prediction.py`

**Steps:**
1. Create a `st.form("prediction_form")` with these inputs:
   - `st.selectbox("Category", categories_list)`
   - `st.selectbox("Platform", platforms_list)`
   - `st.selectbox("Level", levels_list)`
   - `st.number_input("Budget (₹)", min_value=5000, max_value=120000, value=45000, step=1000)`
   - `st.slider("Duration (days)", 7, 90, 30)`
   - `st.slider("Client Rating", 2.5, 5.0, 4.2, 0.1)`
   - `st.slider("Bids Received", 2, 50, 15)`
2. On submit: encode inputs using saved `le_dict`, predict class and probability
3. Display: predicted class + `st.progress(int(prob*100))` as confidence bar

**Answer key:** Input (ML/AI | Upwork | Mid | ₹45,000 | 30d | 4.2 | 15 bids) → **Completed**, **57.5%**

---

### Task 4 (10 pts) — Confidence Colour Coding

**Still in:** `pages/1_🔮_Single_Prediction.py` (extends Task 3)

**Steps:**
1. After the progress bar, add colour-coded result box:
   - `prob >= 0.70` → `st.success("✅ High confidence: Completed")`
   - `0.50 ≤ prob < 0.70` → `st.warning("⚠️ Moderate confidence: Completed")`
   - `prob < 0.50` → `st.error("❌ Low confidence: Not Completed")`
2. Show a 3-column breakdown:
   - Col 1: `st.metric("Prediction", "Completed" or "Not Completed")`
   - Col 2: `st.metric("Confidence", f"{prob*100:.1f}%")`
   - Col 3: `st.metric("Risk", "Low" / "Medium" / "High" based on threshold)`

**Answer key (sample input):** `st.warning` fires (prob=57.5% → moderate band)

---

### Task 5 (10 pts) — Batch Prediction Page

**File:** `pages/2_📊_Batch_Prediction.py`

**Steps:**
1. `st.file_uploader("Upload project CSV", type=["csv"])`
2. When file uploaded:
   - Read with `pd.read_csv()`
   - Show `st.dataframe(df.head(5))` preview
   - Fill nulls with same medians as training
   - Encode categoricals with saved `le_dict`
   - Predict class + probability for all rows
   - Add columns: `predicted_outcome` and `confidence_pct`
3. Show `st.dataframe(results[['project_id','category','platform','budget_inr','predicted_outcome','confidence_pct']])` styled table
4. Show summary metrics: total rows, % predicted Completed, avg confidence

---

### Task 6 (10 pts) — Download Button for Batch Results

**Still in:** `pages/2_📊_Batch_Prediction.py` (extends Task 5)

**Steps:**
1. Convert results DataFrame to CSV bytes: `results.to_csv(index=False).encode('utf-8')`
2. Add `st.download_button()`:
   - `label="⬇ Download Predictions CSV"`
   - `data=csv_bytes`
   - `file_name="freelancehub_predictions.csv"`
   - `mime="text/csv"`
3. Show `st.success(f"✅ {len(results)} predictions ready for download")`

---

### Task 7 (10 pts) — Model Performance Page

**File:** `pages/3_📈_Model_Performance.py`

**Steps:**
1. **Confusion Matrix (Plotly heatmap):**
   - Use `px.imshow([[TN,FP],[FN,TP]], labels=dict(x="Predicted",y="Actual"), ...)` 
   - Known values: TN=0, FP=35, FN=3, TP=62
   - Use `color_continuous_scale="Blues"`
   - Display with `st.plotly_chart(fig, use_container_width=True)`

2. **Feature Importance (Plotly bar chart):**
   - Horizontal bar: features on y-axis, importance on x-axis
   - Color: `color_discrete_sequence=["#1F3864"]`
   - Use `px.bar(fi_df, x='importance', y='feature', orientation='h')`
   - Known top feature: `bids_received` (0.2108)

3. **SHAP Global Bar Chart:**
   - Plotly horizontal bar from mean |SHAP| values
   - Known top SHAP feature: `category` (0.0300)
   - Title: "Mean |SHAP| — What Drives Predictions"

---

### Task 8 (10 pts) — NRA Written Insights

**Add to:** bottom of `pages/3_📈_Model_Performance.py` inside `st.expander("📝 Business Insights")`

Write **2 NRA insights** as `st.markdown()` text:

**NRA 1 — Model Quality:**
> Number: ROC-AUC = [exact value]
> Reason: [why this metric signals poor discrimination]
> Action: [specific next step — name a technique]

**NRA 2 — Top Feature:**
> Number: [top feature] importance = [exact value]
> Reason: [business explanation for why this feature matters]
> Action: [how a freelancer can use this to win more projects]

---

### ⭐ Bonus Task (10★) — SHAP Waterfall for One Prediction

**In:** `pages/1_🔮_Single_Prediction.py`, after the prediction result

**Steps:**
1. Compute SHAP values for the single input row using `shap.TreeExplainer(model)`
2. Build a horizontal bar chart showing each feature's SHAP contribution
3. Color: positive SHAP = green (`#2ECC71`), negative SHAP = red (`#E74C3C`)
4. Title: `"Why this prediction? — SHAP Feature Contributions"`
5. Display with `st.plotly_chart(fig, use_container_width=True)`


---
## 🔑 Section 4 — Answer Key

### Task 1 — Model Numbers
| Metric | Value |
|---|---|
| Dataset rows | 500 |
| Completed in raw data | 325 / 500 (65%) |
| Null fill — `client_rating` median | **3.8** |
| Null fill — `bids_received` median | **28** |
| Null fill — `duration_days` median | **49** |
| Train rows | **400** |
| Test rows | **100** |
| Model accuracy | **62.00%** |
| ROC-AUC | **0.5108** |
| TN / FP / FN / TP | **0 / 35 / 3 / 62** |

### Task 1 — Feature Importance (exact)
| Rank | Feature | Importance |
|---|---|---|
| 1 | bids_received | 0.2108 |
| 2 | budget_inr | 0.2083 |
| 3 | duration_days | 0.2003 |
| 4 | client_rating | 0.1609 |
| 5 | category | 0.1200 |
| 6 | platform | 0.0594 |
| 7 | level | 0.0404 |

### Task 1 — Mean |SHAP| Values
| Rank | Feature | Mean |SHAP| |
|---|---|---|
| 1 | category | 0.0300 |
| 2 | bids_received | 0.0282 |
| 3 | duration_days | 0.0258 |
| 4 | budget_inr | 0.0195 |
| 5 | client_rating | 0.0148 |
| 6 | level | 0.0077 |
| 7 | platform | 0.0077 |

> ⚠️ **Important teaching point:** Feature importance ≠ SHAP global importance.
> RF importances = tree split gain. SHAP = actual contribution to predictions.
> `bids_received` ranks #1 in RF importances but #2 in SHAP. `category` ranks #5 in RF but #1 in SHAP.
> When the client asks "what drives your model?", SHAP is the more trustworthy answer.

### Task 3 — Sample Prediction
| Input | Value |
|---|---|
| Category | ML/AI |
| Platform | Upwork |
| Level | Mid |
| Budget | ₹45,000 |
| Duration | 30 days |
| Client Rating | 4.2 |
| Bids | 15 |
| **Predicted class** | **Completed (1)** |
| **P(Completed)** | **57.5%** |
| **Confidence band** | **Moderate (st.warning fires)** |

### Task 8 — NRA Insights (model answers)

**NRA 1 — Model Quality:**
- **Number:** ROC-AUC = 0.51 — only 1 percentage point above random chance (0.50)
- **Reason:** 65% of projects are "Completed", so the model learned to predict the majority class almost always (TN=0). With no true negatives, it flags every project as Completed.
- **Action:** Retrain with `class_weight='balanced'` in RandomForestClassifier, or use SMOTE oversampling on the minority class. Target AUC ≥ 0.70 before productionising.

**NRA 2 — Top Feature (category, SHAP):**
- **Number:** `category` has mean |SHAP| = 0.030 — the highest among all 7 features per SHAP analysis
- **Reason:** Project type (ML/AI vs Graphic Design) determines whether the model sees the project as completable — possibly because ML/AI projects have higher budgets and longer durations on average
- **Action:** As a freelancer, filter bids to ML/AI and Data Analytics categories where completion probability is highest; avoid Content Writing and SEO projects in this dataset's pattern.


---
## 💻 Section 5 — Full App Code (Answer Key Implementation)

### `build_model.py` — Run once to generate pkl

In [ ]:
# ─── BUILD_MODEL.PY ──────────────────────────────────────────────────────────
# Purpose : Train RandomForest on FreelanceHub data + save pkl bundle
# Method  : Label encoding → 80/20 split → RF(100, max_depth=6) → joblib save
# Run     : python build_model.py   (run ONCE, not inside Streamlit)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib, warnings
warnings.filterwarnings('ignore')

# ── 1. Generate data ─────────────────────────────────────────────────────────
np.random.seed(141)
n = 500
df = pd.DataFrame({
    'project_id':    [f'PJ{1000+i}' for i in range(n)],
    'category':      np.random.choice(['ML/AI','Web Dev','Data Analytics',
                                       'Graphic Design','Content Writing',
                                       'SEO/Digital Marketing'], n),
    'platform':      np.random.choice(['Upwork','Freelancer','Fiverr',
                                       'Toptal','Direct Client'], n,
                                      p=[0.30,0.25,0.25,0.10,0.10]),
    'city':          np.random.choice(['Mumbai','Bangalore','Delhi','Hyderabad',
                                       'Chennai','Pune','Kolkata'], n),
    'level':         np.random.choice(['Entry','Mid','Senior'], n, p=[0.40,0.40,0.20]),
    'budget_inr':    np.random.randint(5000, 120001, n),
    'duration_days': np.random.randint(7, 91, n).astype(float),
    'client_rating': np.round(np.random.uniform(2.5, 5.0, n), 1),
    'bids_received': np.random.randint(2, 51, n).astype(float),
    'outcome':       np.random.choice(['Completed','Cancelled','In Progress'], n,
                                      p=[0.65,0.20,0.15]),
    'month_posted':  np.random.choice(['Jan','Feb','Mar','Apr','May','Jun',
                                       'Jul','Aug','Sep','Oct','Nov','Dec'], n),
})
null_idx = np.random.choice(n, 25, replace=False)
df.loc[null_idx[:10],  'client_rating'] = np.nan
df.loc[null_idx[10:20],'bids_received'] = np.nan
df.loc[null_idx[20:],  'duration_days'] = np.nan

# ── 2. Feature engineering ───────────────────────────────────────────────────
df2 = df.copy()
df2['target'] = (df2['outcome'] == 'Completed').astype(int)

# Medians on full dataset (before split — this is acceptable for imputation)
medians = {
    'client_rating': df2['client_rating'].median(),   # 3.8
    'bids_received': df2['bids_received'].median(),   # 28
    'duration_days': df2['duration_days'].median(),   # 49
}
for col, val in medians.items():
    df2[col].fillna(val, inplace=True)

# Label encode
le_dict = {}
for col in ['category', 'platform', 'level']:
    le = LabelEncoder()
    df2[col + '_enc'] = le.fit_transform(df2[col])
    le_dict[col] = le

feat_cols = ['budget_inr','duration_days','client_rating','bids_received',
             'category_enc','platform_enc','level_enc']

X = df2[feat_cols]
y = df2['target']

# ── 3. Train/test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=141, stratify=y)

# ── 4. Train model ────────────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=141)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(f"Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")   # 62.00%
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")         # 0.5108
print("CM:", confusion_matrix(y_test, y_pred))                   # [[0,35],[3,62]]

# ── 5. Save bundle ────────────────────────────────────────────────────────────
bundle = {
    'model':     model,
    'le_dict':   le_dict,
    'feat_cols': feat_cols,
    'features':  ['budget_inr','duration_days','client_rating','bids_received',
                  'category','platform','level'],
    'medians':   medians,
    'categories': ['ML/AI','Web Dev','Data Analytics','Graphic Design',
                   'Content Writing','SEO/Digital Marketing'],
    'platforms':  ['Upwork','Freelancer','Fiverr','Toptal','Direct Client'],
    'levels':     ['Entry','Mid','Senior'],
}
joblib.dump(bundle, 'day146_model.pkl')
print("✅ Saved: day146_model.pkl")


### `Home.py` — Entry page

In [ ]:
# ─── HOME.PY ─────────────────────────────────────────────────────────────────
# Purpose : App entry point — model summary + key performance metrics
# Run     : streamlit run day146_app/Home.py

import streamlit as st
import joblib

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="FreelanceHub Predictor",
    page_icon="🔮",
    layout="wide"
)

# ── Load model bundle ─────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    # @st.cache_resource: loads once per app session — not on every user click
    return joblib.load("day146_model.pkl")

bundle = load_model()

# ── Header ────────────────────────────────────────────────────────────────────
st.title("🔮 FreelanceHub Outcome Predictor")
st.caption("Month 8, Day 146 — ML Explainability Dashboard | FreelanceHub India (seed=141)")

st.markdown("---")

# ── Model info box ────────────────────────────────────────────────────────────
st.info(
    "This app predicts whether a freelance project will be **Completed** or "
    "**Not Completed**, and explains the prediction using SHAP feature contributions."
)

st.warning(
    "⚠️ **Model Caveat:** ROC-AUC = 0.51 — near-random performance due to class "
    "imbalance (65% Completed). This dashboard demonstrates ML deployment patterns. "
    "A production model would need class balancing before deployment."
)

# ── KPI metrics ───────────────────────────────────────────────────────────────
st.subheader("Model Performance Snapshot")
c1, c2, c3, c4 = st.columns(4)
c1.metric("Accuracy",     "62.00%", help="Correct predictions / total test rows")
c2.metric("ROC-AUC",      "0.51",   help="Area under ROC curve (0.5 = random, 1.0 = perfect)")
c3.metric("Training Rows","400",    help="80% of 500 rows, stratified")
c4.metric("Test Rows",    "100",    help="20% held-out, stratified")

st.markdown("---")

# ── Navigation guide ─────────────────────────────────────────────────────────
st.subheader("📌 App Pages")
st.markdown("""
| Page | What it does |
|---|---|
| 🔮 Single Prediction | Enter one project's details → get prediction + SHAP explanation |
| 📊 Batch Prediction | Upload a CSV → predict all rows → download results |
| 📈 Model Performance | Confusion matrix · Feature importance · SHAP bar chart |
""")


### `pages/1_🔮_Single_Prediction.py`

In [ ]:
# ─── PAGES/1_🔮_SINGLE_PREDICTION.PY ─────────────────────────────────────────
# Purpose : Predict outcome for one project + colour-coded confidence + SHAP
# Methods : st.form, @st.cache_resource, st.progress, st.metric, Plotly SHAP bar

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
import shap
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Single Prediction", page_icon="🔮", layout="wide")
st.title("🔮 Single Project Prediction")
st.caption("Enter project details → instant prediction + SHAP explanation")

# ── Load model ────────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    return joblib.load("day146_model.pkl")

bundle = load_model()
model   = bundle['model']
le_dict = bundle['le_dict']
medians = bundle['medians']

cats  = bundle['categories']
plats = bundle['platforms']
lvls  = bundle['levels']

# ── Input form ────────────────────────────────────────────────────────────────
with st.form("prediction_form"):
    st.subheader("Project Details")
    c1, c2, c3 = st.columns(3)
    with c1:
        category = st.selectbox("Category", cats)
        platform = st.selectbox("Platform", plats)
        level    = st.selectbox("Level",    lvls)
    with c2:
        budget   = st.number_input("Budget (₹)", min_value=5000,
                                   max_value=120000, value=45000, step=1000)
        duration = st.slider("Duration (days)", 7, 90, 30)
    with c3:
        rating   = st.slider("Client Rating", 2.5, 5.0, 4.2, 0.1)
        bids     = st.slider("Bids Received",  2, 50, 15)

    submitted = st.form_submit_button("🔮 Predict")

# ── Prediction logic ──────────────────────────────────────────────────────────
if submitted:
    # Encode inputs
    row = pd.DataFrame({
        'budget_inr':    [budget],
        'duration_days': [float(duration)],
        'client_rating': [rating],
        'bids_received': [float(bids)],
        'category_enc':  [le_dict['category'].transform([category])[0]],
        'platform_enc':  [le_dict['platform'].transform([platform])[0]],
        'level_enc':     [le_dict['level'].transform([level])[0]],
    })

    pred_class = model.predict(row)[0]
    pred_proba = model.predict_proba(row)[0]
    prob       = pred_proba[1]           # P(Completed)
    label      = "Completed" if pred_class == 1 else "Not Completed"

    st.markdown("---")
    st.subheader("Prediction Result")

    # T3 — Metrics + progress bar
    m1, m2, m3 = st.columns(3)
    m1.metric("Prediction",  label)
    m2.metric("Confidence",  f"{prob*100:.1f}%")
    risk = "Low" if prob >= 0.70 else ("Medium" if prob >= 0.50 else "High")
    m3.metric("Risk",        risk)

    st.progress(int(prob * 100))

    # T4 — Colour-coded result box
    if prob >= 0.70:
        st.success(f"✅ High confidence: {label} ({prob*100:.1f}%)")
    elif prob >= 0.50:
        st.warning(f"⚠️ Moderate confidence: {label} ({prob*100:.1f}%)")
    else:
        st.error(f"❌ Low confidence: {label} ({(1-prob)*100:.1f}% chance Not Completed)")

    # ★ BONUS — SHAP waterfall (local explanation)
    st.markdown("---")
    st.subheader("⭐ Why this prediction? — SHAP Feature Contributions")

    explainer  = shap.TreeExplainer(model)
    shap_out   = explainer.shap_values(row)
    if isinstance(shap_out, list):
        sv = shap_out[1][0]
    elif shap_out.ndim == 3:
        sv = shap_out[0, :, 1]
    else:
        sv = shap_out[0]

    feat_labels = ['budget_inr','duration_days','client_rating',
                   'bids_received','category','platform','level']
    shap_df = pd.DataFrame({'feature': feat_labels, 'shap': sv})
    shap_df = shap_df.sort_values('shap')

    colors = ['#E74C3C' if v < 0 else '#2ECC71' for v in shap_df['shap']]

    fig = go.Figure(go.Bar(
        x=shap_df['shap'], y=shap_df['feature'],
        orientation='h', marker_color=colors,
        text=[f"{v:+.3f}" for v in shap_df['shap']], textposition='outside'
    ))
    fig.update_layout(
        title="SHAP Values — Feature Contributions to This Prediction",
        xaxis_title="SHAP Value (positive = pushes toward Completed)",
        yaxis_title="Feature",
        height=400,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption("🟢 Green = pushes prediction toward Completed | 🔴 Red = pushes away from Completed")


### `pages/2_📊_Batch_Prediction.py`

In [ ]:
# ─── PAGES/2_📊_BATCH_PREDICTION.PY ─────────────────────────────────────────
# Purpose : Upload CSV of projects → predict all → styled table → download
# Methods : st.file_uploader, pd.read_csv, joblib, st.download_button

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Batch Prediction", page_icon="📊", layout="wide")
st.title("📊 Batch Prediction")
st.caption("Upload a CSV of projects → predict outcomes for all rows → download results")

# ── Load model ────────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    return joblib.load("day146_model.pkl")

bundle  = load_model()
model   = bundle['model']
le_dict = bundle['le_dict']
medians = bundle['medians']

# ── File upload ───────────────────────────────────────────────────────────────
uploaded = st.file_uploader("Upload project CSV", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)
    st.subheader("Preview (first 5 rows)")
    st.dataframe(df.head(5), use_container_width=True)

    # ── Prepare features ─────────────────────────────────────────────────────
    df2 = df.copy()
    for col in ['client_rating','bids_received','duration_days']:
        if col in df2.columns:
            df2[col].fillna(medians[col], inplace=True)

    for col in ['category','platform','level']:
        if col in df2.columns:
            try:
                df2[col+'_enc'] = le_dict[col].transform(df2[col])
            except ValueError:
                # Handle unseen labels by mapping to most frequent
                mode_enc = le_dict[col].transform([le_dict[col].classes_[0]])[0]
                df2[col+'_enc'] = df2[col].apply(
                    lambda x: le_dict[col].transform([x])[0]
                    if x in le_dict[col].classes_ else mode_enc
                )

    feat_cols = ['budget_inr','duration_days','client_rating','bids_received',
                 'category_enc','platform_enc','level_enc']

    # Check all columns exist
    missing = [c for c in feat_cols if c not in df2.columns]
    if missing:
        st.error(f"Missing columns after encoding: {missing}")
    else:
        X_batch = df2[feat_cols]
        preds   = model.predict(X_batch)
        probas  = model.predict_proba(X_batch)[:,1]

        # ── Build results ─────────────────────────────────────────────────────
        results = df.copy()
        results['predicted_outcome'] = ['Completed' if p==1 else 'Not Completed' for p in preds]
        results['confidence_pct']    = (probas * 100).round(1)

        # ── T5 — Results table ───────────────────────────────────────────────
        st.subheader("Prediction Results")
        display_cols = [c for c in ['project_id','category','platform',
                                    'budget_inr','predicted_outcome','confidence_pct']
                       if c in results.columns]
        st.dataframe(results[display_cols], use_container_width=True)

        # Summary metrics
        n_completed   = (preds == 1).sum()
        avg_conf      = probas.mean() * 100
        s1, s2, s3    = st.columns(3)
        s1.metric("Total Rows",         len(results))
        s2.metric("Predicted Completed", f"{n_completed} ({n_completed/len(results)*100:.0f}%)")
        s3.metric("Avg Confidence",      f"{avg_conf:.1f}%")

        # ── T6 — Download ─────────────────────────────────────────────────────
        csv_bytes = results.to_csv(index=False).encode('utf-8')
        st.download_button(
            label     = "⬇ Download Predictions CSV",
            data      = csv_bytes,
            file_name = "freelancehub_predictions.csv",
            mime      = "text/csv"
        )
        st.success(f"✅ {len(results)} predictions ready for download")

else:
    st.info("Upload a CSV with columns: project_id, category, platform, level, "
            "budget_inr, duration_days, client_rating, bids_received")
    st.caption("Tip: use the FreelanceHub raw data CSV to test batch prediction.")


### `pages/3_📈_Model_Performance.py`

In [ ]:
# ─── PAGES/3_📈_MODEL_PERFORMANCE.PY ────────────────────────────────────────
# Purpose : Confusion matrix + Feature importance + SHAP global + NRA insights
# Methods : Plotly heatmap, px.bar, st.expander, st.markdown

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Model Performance", page_icon="📈", layout="wide")
st.title("📈 Model Performance")
st.caption("Confusion matrix · Feature importance · SHAP global · Business insights")

# ── Load model ────────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    return joblib.load("day146_model.pkl")

bundle = load_model()
model  = bundle['model']

# ── Known test-set performance values (from build_model.py) ─────────────────
TN, FP, FN, TP = 0, 35, 3, 62

# ── T7a — Confusion Matrix ────────────────────────────────────────────────────
st.subheader("Confusion Matrix — Test Set (n=100)")
cm_data = [[TN, FP], [FN, TP]]

fig_cm = px.imshow(
    cm_data,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Not Completed", "Completed"],
    y=["Not Completed", "Completed"],
    color_continuous_scale="Blues",
    text_auto=True
)
fig_cm.update_layout(title="Confusion Matrix", height=350)
st.plotly_chart(fig_cm, use_container_width=True)

st.info(f"**TN={TN} | FP={FP} | FN={FN} | TP={TP}** — "
        "The model predicts almost everything as Completed (TN=0). "
        "This reflects the 65% class imbalance in the dataset.")

# ── T7b — Feature Importance ─────────────────────────────────────────────────
st.subheader("Feature Importance — Random Forest")
fi_data = {
    'feature':    ['bids_received','budget_inr','duration_days','client_rating',
                   'category','platform','level'],
    'importance': [0.2108, 0.2083, 0.2003, 0.1609, 0.1200, 0.0594, 0.0404]
}
fi_df = pd.DataFrame(fi_data).sort_values('importance')

fig_fi = px.bar(
    fi_df, x='importance', y='feature', orientation='h',
    color_discrete_sequence=["#1F3864"],
    title="Feature Importance (Random Forest split gain)",
    labels={'importance': 'Importance Score', 'feature': 'Feature'}
)
fig_fi.update_layout(height=350, plot_bgcolor='white', paper_bgcolor='white')
st.plotly_chart(fig_fi, use_container_width=True)

# ── T7c — SHAP Global Bar ────────────────────────────────────────────────────
st.subheader("Mean |SHAP| — Global Feature Impact")
shap_data = {
    'feature':   ['category','bids_received','duration_days','budget_inr',
                  'client_rating','level','platform'],
    'mean_shap': [0.0300, 0.0282, 0.0258, 0.0195, 0.0148, 0.0077, 0.0077]
}
shap_df = pd.DataFrame(shap_data).sort_values('mean_shap')

fig_shap = px.bar(
    shap_df, x='mean_shap', y='feature', orientation='h',
    color_discrete_sequence=["#2ECC71"],
    title="Mean |SHAP| — What Drives Predictions (SHAP)",
    labels={'mean_shap': 'Mean |SHAP Value|', 'feature': 'Feature'}
)
fig_shap.update_layout(height=350, plot_bgcolor='white', paper_bgcolor='white')
st.plotly_chart(fig_shap, use_container_width=True)

st.warning("**Feature importance ≠ SHAP importance.** "
           "RF importances use split gain; SHAP uses actual prediction contributions. "
           "`bids_received` is #1 in RF importance but #2 in SHAP. "
           "`category` is #5 in RF but #1 in SHAP. Trust SHAP for explainability.")

# ── T8 — NRA Insights ─────────────────────────────────────────────────────────
with st.expander("📝 Business Insights (NRA Format)"):
    st.markdown("""
### NRA 1 — Model Quality

**Number:** ROC-AUC = 0.51 — only 1 percentage point above random chance (0.50)

**Reason:** The dataset has 65% Completed projects. The model learned to predict the majority
class almost exclusively (TN=0 — zero true negatives). With no ability to distinguish
Not Completed projects, the model provides near-zero discrimination value.

**Action:** Retrain using `RandomForestClassifier(class_weight='balanced', ...)` or apply
SMOTE oversampling via `imbalanced-learn`. Target ROC-AUC ≥ 0.70 before deploying to
any client-facing production environment.

---

### NRA 2 — Top Feature (SHAP)

**Number:** `category` has mean |SHAP| = 0.030 — the highest among all 7 features per SHAP analysis

**Reason:** Project category (ML/AI vs Graphic Design vs SEO) is the strongest driver of
predicted completion probability. This likely reflects that higher-budget, longer-duration
categories (ML/AI) are systematically different from commodity categories (Content Writing).

**Action:** As a freelancer bidding on Upwork, concentrate bids in ML/AI and Data Analytics
categories — the model indicates these project types have the highest completion probability,
translating to better client relationships and repeat business.
""")


---
## 🏆 Section 6 — Scoring Rubric

| Task | Description | Points |
|---|---|---|
| T1 | Model training: accuracy=62.00%, AUC=0.5108, correct medians (3.8/28/49), correct split (400/100) | 10 |
| T2 | Home.py: page config, @st.cache_resource, 4 KPI metrics correct, info + warning boxes present | 10 |
| T3 | Single prediction form: all 7 inputs, correct encoding, correct output (Completed, 57.5%) | 10 |
| T4 | Colour coding: st.warning fires for sample (moderate band 50–70%), st.metric shows Risk | 10 |
| T5 | Batch prediction: CSV upload → predict → styled results table → summary metrics correct | 10 |
| T6 | Download button: correct mime type, file name, st.success message | 10 |
| T7 | Performance page: confusion matrix (0/35/3/62), FI bar chart, SHAP bar chart, all exact values | 10 |
| T8 | 2 × NRA insights: exact numbers cited, specific reason, actionable recommendation named | 10 |
| **Total** | | **80** |
| ⭐ Bonus | SHAP waterfall chart on single prediction page (Plotly bar, green/red colouring, correct direction) | 10★ |

---

### Deduction Guide

| Error | Deduction |
|---|---|
| Wrong accuracy / AUC (off by >0.01) | −5 |
| Missing @st.cache_resource (model loads on every click) | −3 |
| NRA insight cites "high" accuracy without noting AUC=0.51 (communication error) | −4 |
| Confusion matrix values wrong | −5 |
| No download button or wrong mime type | −3 |
| NRA number missing or vague ("around 60%") | −3 per insight |
| Bonus chart uses matplotlib instead of Plotly | −5★ |

---

### Interview Framing

**Q: "How do you present an ML model to a non-technical client?"**

> "I build a Streamlit app with three sections: a prediction form so they can enter their
> own data and get instant results, a batch upload page so they can score their entire
> dataset in one click with a downloadable CSV, and a performance dashboard with a
> confusion matrix and SHAP feature importance charts.
>
> The SHAP charts are key — instead of saying 'the model uses these 7 features,'
> I can show the client exactly which features pushed *their specific project* toward
> or away from completion. That's what makes the deliverable defensible in a client review.
>
> For this FreelanceHub model, I'd also flag upfront that ROC-AUC is 0.51 and recommend
> retraining with class balancing — shipping a near-random model without disclosing it
> would destroy client trust."
